# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [9]:
!pip install -q duckdb pandas pyarrow


In [10]:
import duckdb
import pandas as pd
import os
import getpass

In [11]:
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("❌ Invalid Hugging Face token. It must start with 'hf_'.")

print("✅ Token Loaded Successfully")

Paste your Hugging Face READ token (hf_...): ··········
✅ Token Loaded Successfully


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [13]:
con = duckdb.connect()

# Register Hugging Face token
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("✅ DuckDB connected to Hugging Face!")

# Dataset location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Define dataset tables
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Count rows in each table
for name, src in TABLES.items():
    try:
        rows = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
        print(f"{name:25} {rows:,} rows")
    except Exception as e:
        print(f"❌ {name} -> {e}")

✅ DuckDB connected to Hugging Face!
dim_clients               104 rows
dim_content               519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily                78,835,655 rows
fact_daily_sample         11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [14]:
import pandas as pd

clients = con.sql(f"""
    SELECT
        client_hash_id,
        access_profile,
        gsc_data_start,
        ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print(
    "Clients with 12+ months of GSC history:",
    (
        clients["gsc_data_start"]
        <= clients["gsc_data_start"].dropna().max() - pd.Timedelta(days=365)
    ).sum()
)

print("\nFirst 10 Clients:")
display(clients.head(10))

print("\nDataset Information:")
print(clients.info())

print("\nMissing Values:")
print(clients.isnull().sum())

print("\nSummary Statistics:")
display(clients.describe(include="all"))

Clients with 12+ months of GSC history: 4

First 10 Clients:


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   client_hash_id  104 non-null    object        
 1   access_profile  104 non-null    object        
 2   gsc_data_start  67 non-null     datetime64[us]
 3   ga4_data_start  51 non-null     datetime64[us]
dtypes: datetime64[us](2), object(2)
memory usage: 3.4+ KB
None

Missing Values:
client_hash_id     0
access_profile     0
gsc_data_start    37
ga4_data_start    53
dtype: int64

Summary Statistics:


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
count,104,104,67,51
unique,104,5,NaN,NaN
top,client_9958f0a7ae1df715,gsc_and_ga4,NaN,NaN
freq,1,53,NaN,NaN
mean,NaN,NaN,2025-11-17 00:42:59.104477,2026-02-23 05:38:49.411764
min,NaN,NaN,2025-01-27 00:00:00,2025-10-29 00:00:00
25%,NaN,NaN,2025-09-24 00:00:00,2026-02-19 00:00:00
50%,NaN,NaN,2025-11-05 00:00:00,2026-02-20 00:00:00
75%,NaN,NaN,2026-02-19 00:00:00,2026-03-21 12:00:00
max,NaN,NaN,2026-06-02 00:00:00,2026-06-01 00:00:00


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [21]:
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),

windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_last90,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 90 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev90,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_last90,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_last90

    FROM {TABLES['fact_daily']} f
    CROSS JOIN bounds b

    WHERE f.report_date > b.end_d - INTERVAL 180 DAY

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING
        imp_prev90 >= 50
)

SELECT *
FROM windowed
""").df()

print(f"{len(features):,} content items with enough history")

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

135,586 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90
0,client_e547b89c05043229,content_b962dd8115b75719,1657.0,1684.0,2.0,12.171323
1,client_e547b89c05043229,content_d606939ec54831d3,1486.0,1633.0,2.0,38.192883
2,client_e547b89c05043229,content_c06834d9f426a3d6,412.0,376.0,1.0,32.412067
3,client_e547b89c05043229,content_d812cb8e421e99d1,3115.0,4537.0,10.0,9.428185
4,client_e547b89c05043229,content_8e30813d22a7b445,936.0,1039.0,1.0,31.915502


Let's examine the distribution of `imp_prev90` in the `fact_daily_sample` table to determine a more appropriate threshold.

In [20]:
imp_prev90_dist = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily_sample']}
),

calculated_imp_prev90 AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 90 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev90
    FROM {TABLES['fact_daily_sample']} f
    CROSS JOIN bounds b
    WHERE f.report_date > b.end_d - INTERVAL 180 DAY
    GROUP BY f.client_hash_id, f.content_hash_id
)

SELECT imp_prev90
FROM calculated_imp_prev90
WHERE imp_prev90 > 0 -- Only consider items with some impressions
""").df()

print("Distribution of imp_prev90 in fact_daily_sample:")
display(imp_prev90_dist.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distribution of imp_prev90 in fact_daily_sample:


,imp_prev90
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [22]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

# Percentage of impressions coming from the strongest query
qsignals["top_query_share"] = (
    qsignals["top_query_impressions"]
    / qsignals["kept_impressions"]
)

# Merge with features created in Task 3
data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

import numpy as np

# Calculate Click Through Rate (CTR)
data["click_through_rate"] = np.where(
    data["imp_last90"] > 0,
    data["clk_last90"] / data["imp_last90"],
    0
)

print(f"Joined dataset: {len(data):,} rows")

display(data.head())

Joined dataset: 135,586 rows


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,click_through_rate
0,client_e547b89c05043229,content_b962dd8115b75719,1657.0,1684.0,2.0,12.171323,11.0,0.142426,0.727821,56.0,215.0,0.260465,0.001207
1,client_e547b89c05043229,content_d606939ec54831d3,1486.0,1633.0,2.0,38.192883,8.0,0.253701,0.583445,66.0,242.0,0.272727,0.001346
2,client_e547b89c05043229,content_c06834d9f426a3d6,412.0,376.0,1.0,32.412067,2.0,0.271845,0.677184,11.0,21.0,0.523810,0.002427
3,client_e547b89c05043229,content_d812cb8e421e99d1,3115.0,4537.0,10.0,9.428185,12.0,0.067416,0.843660,53.0,277.0,0.191336,0.003210
4,client_e547b89c05043229,content_8e30813d22a7b445,936.0,1039.0,1.0,31.915502,2.0,0.033120,0.925214,27.0,39.0,0.692308,0.001068


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [23]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

# Create target label
# 1 = impressions dropped by more than 20%
# 0 = otherwise
data["is_declining"] = (
    data["imp_last90"] < 0.8 * data["imp_prev90"]
).astype(int)

# Features used for prediction
feature_cols = [
    "imp_prev90",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
]

# Handle missing values in feature_cols by filling with 0
# This is done instead of dropping rows, as NaN often implies absence of a signal for these features.
data[feature_cols] = data[feature_cols].fillna(0)

# Now, model_data will contain all rows, with NaNs in feature_cols replaced by 0.
model_data = data

print(f"Model data after NaN handling: {len(model_data):,} rows")

X = model_data[feature_cols]
y = model_data["is_declining"]

# Group by client so the same client's pages don't appear
# in both train and test sets
groups = model_data["client_hash_id"]

# Split the data
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Baseline accuracy
baseline = max(y_test.mean(), 1 - y_test.mean())

print(f"Baseline Accuracy : {baseline:.3f}")

# Predictions
predictions = model.predict(X_test)

# Evaluation
print(classification_report(
    y_test,
    predictions,
    digits=3
))

Model data after NaN handling: 135,586 rows
Baseline Accuracy : 0.626
              precision    recall  f1-score   support

           0      0.857     0.756     0.803     13250
           1      0.864     0.925     0.893     22154

    accuracy                          0.861     35404
   macro avg      0.860     0.840     0.848     35404
weighted avg      0.861     0.861     0.859     35404



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
